
# Notebook 17 — PDBP Harmonized External Validation

## الهدف

تطبيق النموذج السريري الموحد والمدرّب مسبقًا على بيانات **PPMI** على مجموعة مستقلة من **PDBP** دون:

- إعادة تدريب النموذج.
- إعادة اختيار المتغيرات.
- تعديل المعالجة المسبقة.
- تحسين العتبة باستخدام بيانات PDBP.

## المتغيرات السبعة المتوافقة

1. `ENROLL_AGE`
2. `baseline_NP3TOT`
3. `derived_years_since_PD_diagnosis`
4. `part2_NP2PTOT`
5. `part1_NP1RTOT`
6. `moca_MCATOT`
7. `baseline_NHY`

## النتيجة الخارجية

```text
rapid_progression_q75 = 1
إذا كان التغير السنوي في MDS-UPDRS Part III ≥ 5.0793 نقطة/سنة
```

التحقق الخارجي الأساسي يستخدم العتبة الاحتمالية **0.50**، وتُعرض أيضًا نتيجة ثانوية عند العتبة المقفلة سابقًا **0.45** من Notebook 16.


In [ ]:

# Cell 1 — Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/PPMI_PD_Progression')

MODEL_DIR = (
    PROJECT_DIR
    / 'outputs'
    / 'notebook_16_harmonized_clinical_model_7_features'
)

PDBP_DATA_PATH = (
    PROJECT_DIR
    / 'data'
    / 'external_validation'
    / 'PDBP_external_validation_dataset.csv'
)

OUTPUT_DIR = (
    PROJECT_DIR
    / 'outputs'
    / 'notebook_17_pdbp_external_validation'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_MODEL_PATH = (
    MODEL_DIR / 'harmonized_primary_model_ppmi_7_features.joblib'
)

SENSITIVITY_MODEL_PATH = (
    MODEL_DIR
    / 'harmonized_sensitivity_model_without_baseline_NP3TOT_6_features.joblib'
)

METADATA_PATH = MODEL_DIR / '11_harmonized_model_metadata.json'

print('PDBP data:', PDBP_DATA_PATH)
print('Primary model:', PRIMARY_MODEL_PATH)
print('Sensitivity model:', SENSITIVITY_MODEL_PATH)
print('Output directory:', OUTPUT_DIR)

assert PDBP_DATA_PATH.exists(), f'Missing PDBP file: {PDBP_DATA_PATH}'
assert PRIMARY_MODEL_PATH.exists(), f'Missing primary model: {PRIMARY_MODEL_PATH}'


In [ ]:

# Cell 2 — Imports
import json
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    brier_score_loss,
    roc_curve,
    precision_recall_curve
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

PRIMARY_THRESHOLD = 0.50
LOCKED_SECONDARY_THRESHOLD = 0.45


In [ ]:

# Cell 3 — Utility functions
def binary_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        'n': int(len(y_true)),
        'events': int(y_true.sum()),
        'event_rate_percent': float(100 * y_true.mean()),
        'threshold': float(threshold),
        'ROC_AUC': float(roc_auc_score(y_true, y_prob)),
        'PR_AUC': float(average_precision_score(y_true, y_prob)),
        'balanced_accuracy': float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'sensitivity': float(
            recall_score(y_true, y_pred, zero_division=0)
        ),
        'specificity': float(
            tn / (tn + fp) if (tn + fp) else np.nan
        ),
        'precision': float(
            precision_score(y_true, y_pred, zero_division=0)
        ),
        'F1': float(f1_score(y_true, y_pred, zero_division=0)),
        'Brier_score': float(brier_score_loss(y_true, y_prob)),
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp)
    }

def save_figure(filename):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


In [ ]:

# Cell 4 — Load model metadata and locked pipelines
primary_model = joblib.load(PRIMARY_MODEL_PATH)

sensitivity_model = None
if SENSITIVITY_MODEL_PATH.exists():
    sensitivity_model = joblib.load(SENSITIVITY_MODEL_PATH)

metadata = {}
if METADATA_PATH.exists():
    with open(METADATA_PATH, 'r') as f:
        metadata = json.load(f)

print('Primary model loaded:', type(primary_model))
print('Sensitivity model loaded:', sensitivity_model is not None)

if metadata:
    print('\nNotebook 16 metadata:')
    print(json.dumps(metadata, indent=2))


In [ ]:

# Cell 5 — Load PDBP external-validation dataset
df_raw = pd.read_csv(PDBP_DATA_PATH)

print('Raw PDBP shape:', df_raw.shape)
display(df_raw.head())

print('\nColumns:')
print(df_raw.columns.tolist())


In [ ]:

# Cell 6 — Define locked feature sets and outcome
PRIMARY_FEATURES = [
    'ENROLL_AGE',
    'baseline_NP3TOT',
    'derived_years_since_PD_diagnosis',
    'part2_NP2PTOT',
    'part1_NP1RTOT',
    'moca_MCATOT',
    'baseline_NHY'
]

SENSITIVITY_FEATURES = [
    c for c in PRIMARY_FEATURES
    if c != 'baseline_NP3TOT'
]

OUTCOME_COL = 'rapid_progression_q75'
ID_COL = 'participant_id'

required_columns = [ID_COL] + PRIMARY_FEATURES + [OUTCOME_COL]
missing_columns = [c for c in required_columns if c not in df_raw.columns]

if missing_columns:
    raise KeyError(
        f'Required columns missing from PDBP file: {missing_columns}'
    )

print('All required columns detected: PASS')


In [ ]:

# Cell 7 — Clean data types without changing the locked pipeline
df = df_raw.copy()

numeric_features = [
    'ENROLL_AGE',
    'baseline_NP3TOT',
    'derived_years_since_PD_diagnosis',
    'part2_NP2PTOT',
    'part1_NP1RTOT',
    'moca_MCATOT'
]

for c in numeric_features:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Keep Hoehn & Yahr as string because Notebook 16 treated it as categorical.
df['baseline_NHY'] = df['baseline_NHY'].astype('string')

df[OUTCOME_COL] = pd.to_numeric(
    df[OUTCOME_COL], errors='coerce'
)

# Exclude records lacking the external outcome.
df = df.dropna(subset=[OUTCOME_COL]).copy()
df[OUTCOME_COL] = df[OUTCOME_COL].astype(int)

# Remove accidental duplicate participant rows, if any.
duplicate_ids = int(df[ID_COL].duplicated().sum())
if duplicate_ids:
    print(f'Warning: removing {duplicate_ids} duplicate participant rows.')
    df = df.drop_duplicates(subset=[ID_COL], keep='first').copy()

print('Clean external cohort shape:', df.shape)
print('Unique participants:', df[ID_COL].nunique())
print('\nOutcome distribution:')
display(
    df[OUTCOME_COL]
    .value_counts()
    .rename_axis('class')
    .reset_index(name='n')
)


In [ ]:

# Cell 8 — External-cohort missingness and completeness
missingness = (
    df[PRIMARY_FEATURES]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename('missing_percent')
    .reset_index()
    .rename(columns={'index': 'predictor'})
)

missingness['non_missing_n'] = [
    int(df[c].notna().sum())
    for c in missingness['predictor']
]

display(missingness)

missingness.to_csv(
    OUTPUT_DIR / '01_pdbp_predictor_missingness.csv',
    index=False
)

participant_summary = pd.DataFrame([{
    'external_cohort': 'PDBP-PD',
    'n': len(df),
    'unique_participants': df[ID_COL].nunique(),
    'rapid_progressors': int(df[OUTCOME_COL].sum()),
    'non_rapid_progressors': int(
        len(df) - df[OUTCOME_COL].sum()
    ),
    'rapid_progressor_percent': float(
        100 * df[OUTCOME_COL].mean()
    ),
    'duplicate_participant_rows_removed': duplicate_ids
}])

display(participant_summary)

participant_summary.to_csv(
    OUTPUT_DIR / '02_pdbp_external_cohort_summary.csv',
    index=False
)


In [ ]:

# Cell 9 — Apply the locked PPMI primary model to PDBP
X_external = df[PRIMARY_FEATURES].copy()
y_external = df[OUTCOME_COL].copy()

external_prob = primary_model.predict_proba(X_external)[:, 1]

prediction_table = df[
    [
        ID_COL,
        OUTCOME_COL,
        'baseline_NP3TOT',
        'followup_NP3TOT',
        'annualized_delta_NP3TOT'
    ]
].copy()

prediction_table['predicted_probability'] = external_prob
prediction_table['prediction_threshold_0_50'] = (
    external_prob >= PRIMARY_THRESHOLD
).astype(int)
prediction_table['prediction_threshold_0_45'] = (
    external_prob >= LOCKED_SECONDARY_THRESHOLD
).astype(int)

display(prediction_table.head())

prediction_table.to_csv(
    OUTPUT_DIR / '03_pdbp_external_predictions.csv',
    index=False
)


In [ ]:

# Cell 10 — Primary and locked-secondary external performance
performance_rows = []

primary_metrics = binary_metrics(
    y_external,
    external_prob,
    PRIMARY_THRESHOLD
)
primary_metrics['analysis'] = 'Primary external validation'
primary_metrics['model'] = 'PPMI harmonized clinical model'
performance_rows.append(primary_metrics)

secondary_metrics = binary_metrics(
    y_external,
    external_prob,
    LOCKED_SECONDARY_THRESHOLD
)
secondary_metrics['analysis'] = 'Locked secondary threshold'
secondary_metrics['model'] = 'PPMI harmonized clinical model'
performance_rows.append(secondary_metrics)

external_performance = pd.DataFrame(performance_rows)[
    [
        'analysis', 'model', 'n', 'events',
        'event_rate_percent', 'threshold',
        'ROC_AUC', 'PR_AUC',
        'balanced_accuracy', 'accuracy',
        'sensitivity', 'specificity',
        'precision', 'F1', 'Brier_score',
        'TN', 'FP', 'FN', 'TP'
    ]
]

display(external_performance)

external_performance.to_csv(
    OUTPUT_DIR / '04_pdbp_external_performance.csv',
    index=False
)


In [ ]:

# Cell 11 — Sensitivity model without baseline MDS-UPDRS III
sensitivity_performance = pd.DataFrame()

if sensitivity_model is not None:
    X_external_sens = df[SENSITIVITY_FEATURES].copy()
    sens_prob = sensitivity_model.predict_proba(
        X_external_sens
    )[:, 1]

    sens_rows = []

    for threshold, label in [
        (PRIMARY_THRESHOLD, 'Primary threshold'),
        (
            LOCKED_SECONDARY_THRESHOLD,
            'Locked secondary threshold'
        )
    ]:
        row = binary_metrics(
            y_external,
            sens_prob,
            threshold
        )
        row['analysis'] = label
        row['model'] = (
            'PPMI harmonized sensitivity model '
            'without baseline_NP3TOT'
        )
        sens_rows.append(row)

    sensitivity_performance = pd.DataFrame(sens_rows)

    display(sensitivity_performance)

    sensitivity_performance.to_csv(
        OUTPUT_DIR
        / '05_pdbp_sensitivity_model_external_performance.csv',
        index=False
    )

    sensitivity_predictions = prediction_table[
        [ID_COL, OUTCOME_COL]
    ].copy()
    sensitivity_predictions[
        'predicted_probability_without_baseline_NP3TOT'
    ] = sens_prob

    sensitivity_predictions.to_csv(
        OUTPUT_DIR
        / '06_pdbp_sensitivity_model_predictions.csv',
        index=False
    )
else:
    print('Sensitivity model file not found; analysis skipped.')


In [ ]:

# Cell 12 — ROC curve
fpr, tpr, _ = roc_curve(y_external, external_prob)
roc_auc = roc_auc_score(y_external, external_prob)

plt.figure(figsize=(7, 6))
plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f'PDBP external validation (AUC = {roc_auc:.3f})'
)
plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — PDBP External Validation')
plt.legend(loc='lower right')
save_figure('Figure_1_PDBP_external_ROC_curve.png')


In [ ]:

# Cell 13 — Precision–recall curve
precision, recall, _ = precision_recall_curve(
    y_external, external_prob
)
pr_auc = average_precision_score(
    y_external, external_prob
)
prevalence = y_external.mean()

plt.figure(figsize=(7, 6))
plt.plot(
    recall,
    precision,
    linewidth=2,
    label=f'PDBP external validation (AP = {pr_auc:.3f})'
)
plt.axhline(
    prevalence,
    linestyle='--',
    linewidth=1,
    label=f'Event prevalence = {prevalence:.3f}'
)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision–Recall Curve — PDBP External Validation')
plt.legend(loc='best')
save_figure('Figure_2_PDBP_external_precision_recall_curve.png')


In [ ]:

# Cell 14 — Calibration plot
prob_true, prob_pred = calibration_curve(
    y_external,
    external_prob,
    n_bins=8,
    strategy='quantile'
)

plt.figure(figsize=(7, 6))
plt.plot(
    prob_pred,
    prob_true,
    marker='o',
    linewidth=2,
    label='PDBP external validation'
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--',
    linewidth=1,
    label='Perfect calibration'
)
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed event proportion')
plt.title('Calibration Plot — PDBP External Validation')
plt.legend(loc='best')
save_figure('Figure_3_PDBP_external_calibration_plot.png')

calibration_table = pd.DataFrame({
    'mean_predicted_probability': prob_pred,
    'observed_event_proportion': prob_true
})

calibration_table.to_csv(
    OUTPUT_DIR / '07_pdbp_calibration_table.csv',
    index=False
)


In [ ]:

# Cell 15 — Predicted probability distribution
plt.figure(figsize=(8, 6))

plt.hist(
    external_prob[y_external.to_numpy() == 0],
    bins=20,
    alpha=0.6,
    label='Non-rapid progressors'
)
plt.hist(
    external_prob[y_external.to_numpy() == 1],
    bins=20,
    alpha=0.6,
    label='Rapid progressors'
)

plt.axvline(
    PRIMARY_THRESHOLD,
    linestyle='--',
    linewidth=1,
    label='Threshold 0.50'
)
plt.axvline(
    LOCKED_SECONDARY_THRESHOLD,
    linestyle=':',
    linewidth=1,
    label='Locked threshold 0.45'
)

plt.xlabel('Predicted probability')
plt.ylabel('Participants')
plt.title('Predicted Probability Distribution — PDBP')
plt.legend(loc='best')
save_figure('Figure_4_PDBP_predicted_probability_distribution.png')


In [ ]:

# Cell 16 — Compare internal PPMI and external PDBP performance
internal_perf_path = (
    MODEL_DIR / '05_harmonized_internal_test_performance.csv'
)

comparison_rows = []

if internal_perf_path.exists():
    internal_perf = pd.read_csv(internal_perf_path)

    internal_row = internal_perf.iloc[0].to_dict()
    comparison_rows.append({
        'cohort': 'PPMI internal held-out test',
        'n': np.nan,
        'ROC_AUC': internal_row.get('ROC_AUC', np.nan),
        'PR_AUC': internal_row.get('PR_AUC', np.nan),
        'balanced_accuracy': internal_row.get(
            'balanced_accuracy', np.nan
        ),
        'sensitivity': internal_row.get('sensitivity', np.nan),
        'specificity': internal_row.get('specificity', np.nan),
        'precision': internal_row.get('precision', np.nan),
        'F1': internal_row.get('F1', np.nan),
        'Brier_score': internal_row.get('Brier_score', np.nan)
    })

comparison_rows.append({
    'cohort': 'PDBP external validation',
    'n': len(df),
    'ROC_AUC': primary_metrics['ROC_AUC'],
    'PR_AUC': primary_metrics['PR_AUC'],
    'balanced_accuracy': primary_metrics['balanced_accuracy'],
    'sensitivity': primary_metrics['sensitivity'],
    'specificity': primary_metrics['specificity'],
    'precision': primary_metrics['precision'],
    'F1': primary_metrics['F1'],
    'Brier_score': primary_metrics['Brier_score']
})

performance_comparison = pd.DataFrame(comparison_rows)

display(performance_comparison)

performance_comparison.to_csv(
    OUTPUT_DIR
    / '08_ppmi_internal_vs_pdbp_external_performance.csv',
    index=False
)

metrics_to_plot = [
    'ROC_AUC',
    'PR_AUC',
    'balanced_accuracy',
    'sensitivity',
    'specificity',
    'F1'
]

plot_df = (
    performance_comparison
    .set_index('cohort')[metrics_to_plot]
    .T
)

plot_df.plot(kind='bar', figsize=(10, 6))
plt.ylim(0, 1)
plt.ylabel('Metric value')
plt.xlabel('Performance metric')
plt.title('PPMI Internal vs PDBP External Performance')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Cohort', loc='best')
save_figure('Figure_5_PPMI_internal_vs_PDBP_external_performance.png')


In [ ]:

# Cell 17 — Quality-control checklist
qc = []

def add_qc(check, status, detail):
    qc.append({
        'check': check,
        'status': status,
        'detail': detail
    })

add_qc(
    'PDBP dataset loaded',
    'PASS' if PDBP_DATA_PATH.exists() else 'FAIL',
    str(PDBP_DATA_PATH)
)
add_qc(
    'Locked PPMI model loaded',
    'PASS' if PRIMARY_MODEL_PATH.exists() else 'FAIL',
    str(PRIMARY_MODEL_PATH)
)
add_qc(
    'All seven harmonized predictors present',
    'PASS' if not missing_columns else 'FAIL',
    f'{len(PRIMARY_FEATURES) - len(missing_columns)}/{len(PRIMARY_FEATURES)}'
)
add_qc(
    'External outcome is binary',
    'PASS' if set(y_external.unique()).issubset({0, 1}) else 'FAIL',
    str(sorted(y_external.unique().tolist()))
)
add_qc(
    'Participant IDs unique',
    'PASS' if df[ID_COL].is_unique else 'FAIL',
    f'n={len(df)}, unique={df[ID_COL].nunique()}'
)
add_qc(
    'No model retraining on PDBP',
    'PASS',
    'Only predict_proba was called on the locked PPMI pipeline.'
)
add_qc(
    'No external threshold optimization',
    'PASS',
    'Thresholds 0.50 and locked 0.45 were prespecified.'
)
add_qc(
    'External performance calculated',
    'PASS' if len(external_performance) == 2 else 'FAIL',
    f'rows={len(external_performance)}'
)

qc_df = pd.DataFrame(qc)
display(qc_df)

qc_df.to_csv(
    OUTPUT_DIR / '09_quality_control_checklist.csv',
    index=False
)


In [ ]:

# Cell 18 — Manuscript-ready summary report
summary_lines = [
    'Notebook 17 — PDBP Harmonized External Validation',
    '',
    f'External cohort: PDBP-PD',
    f'Participants: {len(df)}',
    f'Rapid progressors: {int(y_external.sum())} '
    f'({100 * y_external.mean():.2f}%)',
    f'Harmonized predictors: {len(PRIMARY_FEATURES)}',
    '',
    'Primary external validation at threshold 0.50:',
    f'ROC-AUC: {primary_metrics["ROC_AUC"]:.4f}',
    f'PR-AUC: {primary_metrics["PR_AUC"]:.4f}',
    f'Balanced accuracy: '
    f'{primary_metrics["balanced_accuracy"]:.4f}',
    f'Sensitivity: {primary_metrics["sensitivity"]:.4f}',
    f'Specificity: {primary_metrics["specificity"]:.4f}',
    f'Precision: {primary_metrics["precision"]:.4f}',
    f'F1: {primary_metrics["F1"]:.4f}',
    f'Brier score: {primary_metrics["Brier_score"]:.4f}',
    '',
    'Locked secondary threshold 0.45:',
    f'Balanced accuracy: '
    f'{secondary_metrics["balanced_accuracy"]:.4f}',
    f'Sensitivity: {secondary_metrics["sensitivity"]:.4f}',
    f'Specificity: {secondary_metrics["specificity"]:.4f}',
    f'Precision: {secondary_metrics["precision"]:.4f}',
    f'F1: {secondary_metrics["F1"]:.4f}',
    '',
    'Interpretation:',
    (
        'This analysis is a true external validation because the '
        'PPMI pipeline was applied to PDBP without retraining, '
        'feature reselection, preprocessing refitting, or '
        'external threshold optimization.'
    )
]

summary_text = '\n'.join(summary_lines)
print(summary_text)

with open(
    OUTPUT_DIR / '10_notebook_17_summary_report.txt',
    'w'
) as f:
    f.write(summary_text)



## المخرجات الرئيسية

داخل:

```text
PPMI_PD_Progression/outputs/notebook_17_pdbp_external_validation/
```

سيُنشئ الدفتر:

- `01_pdbp_predictor_missingness.csv`
- `02_pdbp_external_cohort_summary.csv`
- `03_pdbp_external_predictions.csv`
- `04_pdbp_external_performance.csv`
- `05_pdbp_sensitivity_model_external_performance.csv`
- `07_pdbp_calibration_table.csv`
- `08_ppmi_internal_vs_pdbp_external_performance.csv`
- `09_quality_control_checklist.csv`
- `10_notebook_17_summary_report.txt`
- خمسة رسوم جاهزة للاستخدام في المخطوطة

> لا تُعدّل النموذج أو العتبات بناءً على نتائج PDBP.
